# TN1 — DS-TCN 192 kênh, tầm nhìn 61, không chuẩn hoá, dropout 0,2

## Khác bản DS-TCN-64 gốc của TN1 ở đâu

Năm chỗ, không phải một:

| | DS-TCN-64 gốc (TN1) | cấu hình này |
|---|---|---|
| channels | 64 | **192** |
| số khối | 6 | **4** |
| **tầm nhìn** | **253**, phủ trọn 200 mẫu | **61**, thấy 61/200 |
| chuẩn hoá | BatchNorm | **không có** |
| dropout | 0, `nn.Dropout1d` | **0,2, `nn.Dropout`** |
| tham số | 56.281 | **307.801** |

Cột phải là kiến trúc DS-TCN do nhóm tối ưu riêng cho bài toán này, dựng lại
nguyên bản. Cột trái là cấu hình TN1 hiện có, bám Bai et al. 2018 và Howard et
al. 2017.

Vì đổi nhiều thứ cùng lúc nên **không quy được kết quả cho riêng thứ nào**.

## Hai notebook chạy song song

| notebook | channels | tham số | thời gian |
|---|---:|---:|---|
| `..._c192.ipynb` | 192 | 307.801 | **~6 giờ** |
| `..._c64.ipynb` | 64 | 37.081 | **~2,5 giờ** |

Hai tệp nén khác tên nên chạy cùng lúc ở hai phiên Colab không đè nhau. Đặt
cạnh nhau, chúng trả lời: ở kiến trúc này, 192 kênh có hơn 64 kênh không.

## Kiến trúc dựng lại nguyên bản, chế độ huấn luyện thì không

Bản gốc dùng loss hỗn hợp, learning rate và weight decay riêng. Ở đây giữ nguyên
giao thức TN1 — MSE, Adam lr 1e-4, weight decay 0, batch 64, 20 epoch,
`corr` 0,9 — để so được với các cấu hình TN1 khác. Kết quả nói về **kiến trúc**,
không tái lập kết quả bản gốc.

## Hai tuỳ chọn phải bổ sung vào code

**`--norm none`.** Trước đây chỉ nhận `batch` hoặc `weight`. Kiến trúc này
**không có lớp chuẩn hoá nào** — đó chính là 3.072 tham số chênh lệch từng thấy,
bằng 8 lớp BatchNorm × 384.

**`--dropout_kind element`.** TN1 dùng `nn.Dropout1d`, xoá cả một kênh, theo
"spatial dropout" ở Bai mục 3.4. Kiến trúc này dùng `nn.Dropout` thường. Hai
loại chỉ khác nhau khi `dropout > 0`, mà mọi cấu hình TN1 để 0, nên chưa từng
lộ ra.

## Đã đối chiếu với bản gốc, khớp từng bit

Dựng lại nguyên văn `CausalDSConv`, `TCNBlock`, `ForecastTCN`, chép trọng số
sang bản này rồi so: cùng số tham số, cùng hình dạng mọi tensor, **đầu ra giống
hệt, chênh lớn nhất 0.000e+00**.

Kiểm thêm bằng thực nghiệm: **đổi 139 mẫu đầu của cửa sổ không làm đổi đầu ra** —
đúng tầm nhìn 61.

## Tầm nhìn 61 — đọc kỹ chỗ này

Cửa sổ vào 200 mẫu, model chỉ thấy **61 mẫu gần nhất**, bỏ qua 70 phần trăm đầu
cửa sổ. Ở 50 Hz thì 61 mẫu là **1,2 giây**, trong khi một nhịp thở khoảng 4
giây. Model không nhìn đủ một chu kỳ thở.

## 1. Chuẩn bị Colab

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tải mã nguồn. Hai cờ `--norm none` và `--dropout_kind element` chỉ có ở bản
mới, nên ô này phải chạy được thì mới đi tiếp.

**Ô này xoá `/content/UWB_RADAR`, tức xoá luôn `runs/` cục bộ.** Mỗi fold
`run_cv.py` tự nén sang Drive nên không mất hẳn, nhưng phiên mới train lại từ
fold đầu.

In [2]:
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : 71aabd6
commit MobiVital : 4319731 (đã ghim)
GPU              : Tesla T4, 15360 MiB


Lấy `by_user/` và `windows/` từ Drive.

In [3]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


Khôi phục kết quả đã có từ Drive.

**Ô clone ở trên xoá `runs/`, nên không có bước này thì mọi fold đã chạy sẽ bị
train lại.** `run_cv.py` biết bỏ qua fold đã xong, nhưng nó chỉ đọc được
`runs/` cục bộ — mà thư mục đó vừa bị xoá.

Chưa có kết quả cũ thì ô này không in gì, chạy tiếp bình thường.

In [4]:
# Khôi phục kết quả đã chạy trước khi phiên bị ngắt.
#
# Mỗi tệp nén chứa một bản runs/<thực nghiệm>/summary.csv của riêng nó. Giải
# hết vào cùng một thư mục runs/ thì tệp giải sau ĐÈ summary.csv của tệp trước.
# Mà sorted() xếp "..._a0_corr0.9..." đứng SAU "..._a0.9_corr0.9...", vì trong
# bảng mã ký tự "_" lớn hơn "." — nên bản ít dòng nhất lại là bản đè cuối cùng.
# Sửa: mỗi tệp nén giải vào một thư mục tạm riêng, gộp mọi dòng lại rồi mới ghi
# runs/summary.csv một lần. Thứ tự giải nén không còn ảnh hưởng gì nữa.
import csv, glob, os, shutil, subprocess, tempfile

MAU_ZIP = "/content/drive/MyDrive/mobivital/tn1_*c192*.zip"

rows, seen = [], set()

def collect(summary_path):
    for r in csv.DictReader(open(summary_path)):
        key = (r.get("experiment"), r.get("run_id"))
        if key not in seen:
            seen.add(key)
            rows.append(r)

for f in sorted(glob.glob(MAU_ZIP)):
    tmp = tempfile.mkdtemp()
    subprocess.run(["unzip", "-oq", f, "-d", tmp], check=True)
    for s in glob.glob(tmp + "/*/summary.csv"):
        collect(s)
        os.remove(s)   # gộp xong thì bỏ, để bước chép dưới không đè nhau nữa
    # Checkpoint và curve.csv nằm trong thư mục riêng của từng lần chạy, tên
    # không trùng nhau, nên chép chồng lên runs/ là an toàn.
    for d in os.listdir(tmp):
        shutil.copytree(tmp + "/" + d, "runs/" + d, dirs_exist_ok=True)
    shutil.rmtree(tmp)

# Dòng đã sinh ra trong chính phiên này cũng phải giữ lại.
if os.path.exists("runs/summary.csv"):
    collect("runs/summary.csv")

if rows:
    # run_cv.py tra runs/summary.csv, còn tệp nén chỉ có runs/<thực nghiệm>/summary.csv
    cols = []
    for r in rows:
        for k in r:
            if k not in cols:
                cols.append(k)
    with open("runs/summary.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=cols, restval="")
        w.writeheader()
        w.writerows(rows)
print("khôi phục", len(rows), "dòng vào runs/summary.csv")

khôi phục 15 dòng vào runs/summary.csv


## 2. Kiểm bản cài đặt

Bảy phép kiểm. Ba phép nhắm đúng chỗ vừa bổ sung:

    mục 5   KHÔNG có lớp chuẩn hoá nào
    mục 6   dùng nn.Dropout chứ không phải nn.Dropout1d
    mục 7   in tầm nhìn 61 và cảnh báo mất phần đầu cửa sổ

Số tham số phải ra đúng **307.801**.

**Phải đọc kết quả ô này trước khi chạy ô sau.** Dòng cuối phải là
`TẤT CẢ ĐẠT`. Nếu là `DỪNG — bản cài đặt có vấn đề` thì đừng bấm tiếp, vì
notebook không tự dừng và sẽ train hàng giờ cho một model sai.

In [5]:
!python scripts/check_model.py --model ds_tcn --channels 192 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element

Kiểm model: ds_tcn

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   36/36 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   307801

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng TCN
   0 lớp BatchNorm1d, WeightNorm: False
   KHÔNG có lớp chuẩn hoá nào, đúng cấu hình đã chọn          đạt
   cũng không có WeightNorm                                   đạt

6. Loại dropout
   0 lớp Dropout1d (xoá cả kênh), 8 lớp Dropout (xoá từng phần tử)
   dùng nn.Dropout, xoá từng phần tử                          đạt

7. Tầm nhìn so với cửa sổ vào
   kernel 3, 4 khối -> tầm nhìn 61, cửa sổ vào 200
   CHÚ Ý: chỉ thấy 61/200 mẫu gần nhất, mất 70% đầu cửa sổ
   tầm nhìn ngắn hơn cửa sổ — có chủ ý, không phải lỗi        đạt

TẤT C

## 3. Chạy 4 fold, 3 seed

Tên cấu hình: `ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed<N>`.

Hậu tố `_none` ghi cách chuẩn hoá, `_dpel` ghi loại dropout. Cả hai chỉ xuất
hiện khi khác mặc định nên tên các lần chạy trước không đổi.

Mỗi seed khoảng **2 giờ**, ba seed khoảng **6 giờ**. Sau mỗi fold script tự nén
sang Drive; ngắt phiên giữa chừng thì chạy lại ô này, fold đã xong được bỏ qua.

Muốn thăm dò trước thì chạy riêng ô seed 0, thấy đáng thì chạy nốt hai ô sau.

In [6]:
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 192 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 0

thực nghiệm tn1  -> runs/tn1/
cấu hình ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
   đã có kết quả 0.7998 — bỏ qua, không train lại
runs/tn1/  ->  runs/tn1_ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0.zip   (13.8 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-09 06:14   tn1/
        0  2026-09-07 15:53   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 16:08   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 16:23   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 16:37   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 16:49   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_AB/
        0  2026-09-07 17:04   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_CE/
        0  

In [7]:
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 192 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 1

thực nghiệm tn1  -> runs/tn1/
cấu hình ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
   đã có kết quả 0.7934 — bỏ qua, không train lại
runs/tn1/  ->  runs/tn1_ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1.zip   (13.8 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-09 06:14   tn1/
        0  2026-09-07 15:53   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 16:08   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 16:23   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 16:37   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 16:49   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_AB/
        0  2026-09-07 17:04   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_CE/
        0  

In [8]:
!python scripts/run_cv.py --experiment tn1 --model ds_tcn --channels 192 \
    --kernel_size 3 --n_blocks 4 --dropout 0.2 --norm none --dropout_kind element --seed 2

thực nghiệm tn1  -> runs/tn1/
cấu hình ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed2
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
   đã có kết quả 0.7971 — bỏ qua, không train lại
runs/tn1/  ->  runs/tn1_ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed2.zip   (13.8 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-09 06:14   tn1/
        0  2026-09-07 15:53   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 16:08   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 16:23   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 16:37   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 16:49   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_AB/
        0  2026-09-07 17:04   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_CE/
        0  

## 4. Cất kết quả

Tên nén riêng, không đè notebook kia.

In [9]:
!python scripts/save_results.py tn1 --out tn1_ds_tcn_rf61_c192

runs/tn1/  ->  runs/tn1_ds_tcn_rf61_c192.zip   (13.8 MB)
   15 dòng metric trong summary.csv

Bên trong:
        0  2026-09-09 06:14   tn1/
        0  2026-09-07 15:53   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 16:08   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 16:23   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 16:37   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_KL/
        0  2026-09-07 16:49   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_AB/
        0  2026-09-07 17:04   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_CE/
        0  2026-09-07 17:19   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_DF/
        0  2026-09-07 17:33   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed1_val_KL/
        0  2026-09-07 17:46   tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed2_val_AB/
        0  202

## 5. Đường hội tụ

Cấu hình này là cấu hình TN1 **đầu tiên dùng dropout khác 0**. Xem `train_mse`
có còn giảm ở epoch 19 không, và `pearson` có đi cùng chiều không.

In [10]:
!cut -d, -f1,2,3 runs/tn1/ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9_seed0_val_AB/curve.csv | tail -6

14,0.021506699455903677,0.5713883792139163
15,0.02139473840080751,0.5728496124175305
16,0.021264110716420807,0.5755487876676739
17,0.021198776085635893,0.5758248207548874
18,0.021131420666213502,0.5777464558364479
19,0.02099776897587341,0.5791048935174147


## 6. Bảng so

`compare_cv` chỉ thấy cấu hình của phiên này vì ô clone đã xoá `runs/`. Bảng đủ
dựng ở `TN1_final_evaluation.ipynb`.

Mốc từ TN1:

| | tham số | tầm nhìn | cv_mean |
|---|---:|---:|---:|
| LSTM-352 | 1.502.713 | — | 0,7570 ± 0,0041 |
| LSTM-67 | 56.908 | — | 0,7532 ± 0,0020 |
| DS-TCN-64 | 56.281 | 253 | 0,7421 ± 0,0007 |

`seed_std` của tám cấu hình TN1 trải từ 0,0007 tới 0,0108 — mỗi kiến trúc một
khác, không mượn của nhau được. Có đủ ba seed rồi mới tính `seed_std` của chính
cấu hình này.

In [11]:
!python scripts/compare_cv.py --experiment tn1


BẢNG 1 — cv_score, thực nghiệm tn1
cấu hình                         tham số  seed    cv_mean  seed_std  fold_std   từng seed
--------------------------------------------------------------------------------------------------------------
ds_tcn_c192_k3_n4_none_do0.2_dpel_mse_corr0.9    307801     3   0.747955  0.012189  0.081731   s0 0.7575  s1 0.7522  s2 0.7342

cv_mean  = trung bình cv_score của các seed. cv_score của một seed là
           trung bình điểm macro 4 fold; macro = trung bình theo NGƯỜI.
seed_std = dao động giữa các seed. Chênh lệch giữa hai cấu hình nhỏ hơn
           số này thì chưa kết luận được.
fold_std = dao động giữa 4 fold, trung bình trên các seed. Nói dữ liệu
           giữa các người khác nhau ra sao, KHÔNG dùng để so cấu hình.



## 7. Ngắt phiên

Kết quả đã nén sang Drive ở mục 4 nên ngắt ở đây không mất gì.

In [ ]:
from google.colab import runtime
runtime.unassign()